<div align="center">

# PPR-opt — reusable inventory

### Build PPRs from the deposited GRASP module kit instead of ordering new DNA

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SyedZainAliShah/clippr/blob/main/notebooks/PPROPT_inventory.ipynb)

**iGEM Marburg 2026**

</div>

---

**Use this notebook if you hold the kit.** If you want a PPR designed and synthesised from
scratch, use [PPROPT_designer.ipynb](https://colab.research.google.com/github/SyedZainAliShah/clippr/blob/main/notebooks/PPROPT_designer.ipynb) instead — the other route, which
needs no kit.

Recode the kit once for your host, then *compile* targets from it: the DNA is ordered once
and reused. The kit can only spell the targets its modules cover, which is the trade you are
making against full synthesis.

**You need Supplementary Table S1** (`.xlsx`). It holds the module insert sequences and is not
distributed with this package — see `NOTICE.md`. Upload it at step 1. Every step after the
recode works from a *saved inventory* carrying its own sequences, so Table S1 is needed once.

**Run the cells top to bottom.** Each step depends on the one before it.

In [ ]:
#@title Setup — install PPR-opt {display-mode: "form"}
#@markdown Installs PPR-opt, or updates it when this notebook is newer than the copy you
#@markdown already have. Safe to re-run. About a minute the first time, seconds after that.
#@markdown
#@markdown **If it fails:** run it again — a dropped network call is the usual cause. If it
#@markdown reports that it updated, use **Runtime → Restart session** and run it once more:
#@markdown the old copy stays loaded in memory until you do. Every cell below needs this one
#@markdown to have succeeded, so do not skip past a red error here.
REPO = "SyedZainAliShah/clippr"

import importlib.metadata as _md, json as _json, sys, urllib.request


def _installed():
    """(kind, commit) for the copy already on this machine, without importing it."""
    try:
        info = _json.loads(_md.distribution("ppropt").read_text("direct_url.json") or "{}")
    except Exception as exc:
        return ("absent", None) if isinstance(exc, _md.PackageNotFoundError) else ("other", None)
    if "vcs_info" in info:
        return "github", info["vcs_info"].get("commit_id")
    if "dir_info" in info or str(info.get("url", "")).startswith("file:"):
        return "local", None
    return "other", None


def _latest():
    """The current main commit, or None when GitHub cannot be reached."""
    try:
        req = urllib.request.Request(f"https://api.github.com/repos/{REPO}/commits/main",
                                     headers={"Accept": "application/vnd.github.sha"})
        with urllib.request.urlopen(req, timeout=10) as r:
            return r.read().decode().strip()
    except Exception:
        return None


# A bare `import ppropt` succeeding is not evidence that the copy is current. An older build
# imports fine and then fails deep in a later cell on a key it never returned, which reads as
# a bug in the notebook. The version string is static, so the commit is what distinguishes
# them. A local checkout is left alone: it is someone working on the package.
_kind, _have = _installed()
_want = _latest()
#: Whether a previous cell already imported it. Reinstalling cannot replace a module object
#: that is already in memory, and the commit check above reads the disk, so "current" says
#: nothing about what this kernel is actually running.
_already = "ppropt" in sys.modules

if _kind == "local":
    _msg = "local checkout in use, left alone"
elif _kind == "absent" or (_kind == "github" and _want and _have != _want):
    token = None
    try:
        from google.colab import userdata          # private-repo fallback
        token = userdata.get("GITHUB_TOKEN")
    except Exception:
        pass
    url = (f"git+https://{token}@github.com/{REPO}.git" if token
           else f"git+https://github.com/{REPO}.git")
    if _kind == "absent":
        %pip install --quiet $url
        import ppropt                               # fail here, not three cells later
        _msg = f"installed ppropt @ {(_want or '?')[:7]}"
    elif "ppropt" in sys.modules:
        # Already imported: reinstalling cannot replace the module object in memory, so the
        # only honest thing to report is that a restart is still required.
        %pip install --quiet --force-reinstall --no-deps $url
        _msg = (f"updated to {(_want or '?')[:7]} — now use Runtime > Restart session and "
                f"run this cell again; the old copy is still loaded until you do")
    else:
        # Dependencies are already present; a newly added one fails loudly on import.
        %pip install --quiet --force-reinstall --no-deps $url
        import ppropt
        _msg = f"updated to {(_want or '?')[:7]}"
elif _kind == "other" or not _have:
    _msg = "ppropt installed, but not from git, so it cannot be checked against main"
elif _want is None:
    _msg = f"ppropt @ {_have[:7]}, could not reach GitHub to check it is current"
else:
    _msg = f"ppropt @ {_have[:7]}, current with {REPO}@main"
    if _already:
        _msg += ("  —  already imported in this session; if it was updated since, "
                 "Runtime > Restart session")

from IPython.display import HTML, display
display(HTML(
    f'<div style="border-left:3px solid #1a7f5a;padding:.5em .9em;'
    f'font-family:ui-monospace,monospace;font-size:13px;opacity:.85">{_msg}</div>'))

In [ ]:
#@title Setup check — does this build have the inventory workflow? {display-mode: "form"}
#@markdown Checks this notebook can run on the copy of PPR-opt you have installed.
#@markdown
#@markdown **Green?** Carry on. **Red?** Run the Setup cell above again, then
#@markdown **Runtime → Restart session**, then Setup once more.
import importlib.util

_needed = ["ppropt.inventories", "ppropt.workflow", "ppropt.ordering", "ppropt.substrates"]
_missing = [m for m in _needed if importlib.util.find_spec(m) is None]

from IPython.display import HTML, display

if _missing:
    display(HTML(
        '<div style="border-left:3px solid #a8402c;padding:.6em 1em;font-size:13px;'
        'line-height:1.55">The installed PPR-opt does not carry the reusable-inventory '
        'workflow.<br>Missing: <code>' + ", ".join(_missing) + '</code><br><br>'
        'Nothing below this cell will run until the published build includes it. The '
        'designer notebook is unaffected.</div>'))
else:
    display(HTML(
        '<div style="border-left:3px solid #1a7f5a;padding:.6em 1em;'
        'font-family:ui-monospace,monospace;font-size:13px;opacity:.85">'
        'inventory workflow available</div>'))

In [ ]:
#@title Before you start — your host  (edit this) {display-mode: "form"}
#@markdown The host the kit is recoded for. The genetic code follows automatically —
#@markdown nuclear hosts use table 1, chloroplasts table 11.
organism = "c_reinhardtii_nuclear"  #@param ["c_reinhardtii_nuclear", "c_reinhardtii_chloroplast", "e_coli", "s_cerevisiae", "a_thaliana_nuclear", "n_tabacum_chloroplast"]

#@markdown Optional: your own codon usage as a `codon,frequency` CSV or a CDS FASTA.
#@markdown Upload it with the folder icon in the sidebar and put the filename here.
codon_table_file = ""  #@param {type:"string"}

#@markdown Optional: the genetic code for a table you supplied. Leave 0 to inherit from the
#@markdown host. Set 11 for anything organellar.
genetic_code_override = 0  #@param {type:"integer"}

---

## Step 1 — load the deposited kit

In [ ]:
#@title Upload Supplementary Table S1 {display-mode: "form"}
#@markdown Run this and choose **Supplementary Table S1** (`.xlsx`) from your computer. It
#@markdown holds the module insert sequences and is not distributed with this package --
#@markdown see `NOTICE.md`.
#@markdown
#@markdown The folder icon in the sidebar does the same job. Either way the file lives on
#@markdown this runtime only: Colab deletes it when the runtime recycles, so it is needed
#@markdown once per session, not once ever.
#@markdown
#@markdown Already have it here? This cell says so and does nothing.
from pathlib import Path as _Path

_WANTED = "Table S1.xlsx"

if _Path(_WANTED).is_file():
    print(f"{_WANTED} is already here -- nothing to upload")
else:
    try:
        from google.colab import files as _files
    except ImportError:
        print(f"not running in Colab. Put {_WANTED} next to this notebook, or set the")
        print("table_s1 field in Step 1 to wherever it already is.")
    else:
        for _name, _data in _files.upload().items():
            print(f"uploaded {_name}  ({len(_data):,} bytes)")
            if _name != _WANTED:
                print(f"   note: Step 1 looks for {_WANTED!r}. Either rename this file or")
                print(f"   set Step 1's table_s1 field to {_name!r}.")

In [ ]:
#@title Step 1 — load the deposited kit {display-mode: "form"}
#@markdown Upload **Supplementary Table S1** (`.xlsx`), then put its filename here.
table_s1 = "Table S1.xlsx"  #@param {type:"string"}

import json as _json
from pathlib import Path

from ppropt import inventories as inv
from ppropt import workflow as w

# Resolves the codon table from the host form above. It cannot borrow the name `table`:
# that name is used elsewhere for a dataframe
# to a display DataFrame, so every call here was handed a DataFrame and raised. The name is
# distinct so a later display cell cannot shadow it again.
from ppropt import constants as _C
from ppropt.design import _resolve_codon_table as _resolve


def done(result):
    """The result if the task succeeded, else None, with the reasons printed.

    For the two tasks that refuse *without writing an artefact*: `explore_interfaces` (an
    empty front has nothing to select) and `select_interface` (it returns before publishing
    anything the verifier rejected). Reaching into `.artefacts[...]` after one of those
    turned a designed refusal into `KeyError: 'inventory'`, which reads like a broken
    notebook rather than the explanation it actually is.

    Not for `recode_for_host`, `load_and_compile` or `order_items_for`. Those write their
    artefact either way and report `ok=False` for a *partial* result -- one infeasible
    module, one unspellable target, one substrate outside the synthesis contract. Nulling
    those would throw away 41 good modules because of the 42nd. They guard on the artefact
    being present instead, and print what fell short.
    """
    if result is None or result.ok:
        return result
    for failure in result.failures:
        print(f"   {failure.get('stage', '?')}: {failure.get('reason', failure)}")
    print()
    print("Nothing was written. Fix the above and re-run this cell; later steps will stay")
    print("skipped until it succeeds.")
    return None

_host_code = genetic_code_override or _C.ORGANISMS[organism][1]
host_codon_table = _resolve(codon_table_file or None, organism, _host_code)
print(f"codon table for {organism}, genetic code {_host_code}: "
      f"{len(host_codon_table)} amino acids")

deposited = None
if not Path(table_s1).is_file():
    print(f"{table_s1} not found. Run the upload cell just above this one, or use the")
    print("folder icon in the sidebar. Nothing below this step will run without it.")
else:
    deposited = inv.load_deposited(table_s1)
    print(f"loaded {len(deposited)} modules  |  version {deposited.version}")
    print(f"structural problems: {inv.validate(deposited) or 'none'}")

    notes = inv.synthesis_profile_notes(deposited)
    if notes:
        print()
        print("Modules outside PPR-opt's synthesis profile, before you order anything:")
        for module_id, why in notes.items():
            print(f"   {module_id}: {'; '.join(why)}")
        print()
        print("That limit is PPR-opt's engineering choice, not a published rule, so this")
        print("is a statement about fit -- not a defect in the deposited kit.")

## Recode the kit for your host

Every module is rewritten to use your host's preferred codons while **the protein, the length
and both four-base interfaces stay exactly as they were**. A recoded module therefore drops
straight into an assembly built from unrecoded neighbours.

A module that cannot be improved keeps its original sequence and is reported as *unchanged*.
That is a delivered module, not a gap.

In [ ]:
#@title Step 2 — recode for your host, interfaces frozen {display-mode: "form"}
label = "my-host"  #@param {type:"string"}
candidate_seeds = 4  #@param {type:"slider", min:1, max:8, step:1}
recode_seconds = 600  #@param {type:"integer"}

recoded = None
if deposited is not None:
    recoded = w.recode_for_host(table_s1, host_codon_table, "out/inventory", label=label,
                                seeds=candidate_seeds, wall_seconds=recode_seconds)
    print(recoded.summary)
    for failure in recoded.failures:
        print(f"   {failure['module']}: {failure['reason']}")

if recoded is not None and "inventory" in recoded.artefacts:
    print(f"saved to {recoded.artefacts['inventory']}")

    report = _json.loads(Path(recoded.artefacts["report"]).read_text(encoding="utf-8"))
    scored = [o for o in report["objectives"].values() if "cai_after" in o]
    if scored:
        before = sum(o["cai_before"] for o in scored) / len(scored)
        after = sum(o["cai_after"] for o in scored) / len(scored)
        print()
        print(f"mean CAI {before:.4f} -> {after:.4f} over {len(scored)} modules")
        print()
        print("CAI predicts how well codons match the host. It is not a measurement of")
        print("expression, and a large CAI gain is not a proportional expression gain.")

## Compile targets from the inventory

Give it targets; it returns the module list, the assembled product and every junction. A
target the kit cannot spell is listed with the reason why, and the others still run.

The product is **joined module inserts only** — no acceptor backbone, so it is not an
expression construct.

In [ ]:
#@title Step 3 — compile targets {display-mode: "form"}
inventory_targets = "AAAAUGUGG, UUACACGUGCGUAC"  #@param {type:"string"}
fusion_site = "AATG"  #@param ["AATG", "AGGT"]

wanted = [t.strip().upper().replace("T", "U")
          for t in inventory_targets.split(",") if t.strip()]
compiled = None
if recoded is not None:
    compiled = w.load_and_compile(recoded.artefacts["inventory"], wanted,
                                  "out/compiled", fusion_site=fusion_site)
    print(compiled.summary)
    for failure in compiled.failures:
        print(f"   could not build {failure['target']}: {failure['reason']}")

# Not `compiled.ok`: one unspellable target makes this False while the others still
# compiled and the artefact still exists. The artefact's presence is the real condition.
if compiled is not None and "compiled_targets" in compiled.artefacts:
    products = _json.loads(
        Path(compiled.artefacts["compiled_targets"]).read_text(encoding="utf-8"))
    for name, product in products.items():
        print(f"   {name}: {product['product_nt']} nt, "
              f"{len(product['modules'])} modules, {product['reactions']} reaction(s)")

## Optimise the inventory as a collection

Recoding treats each module alone, so it cannot see what the modules share *with each other*.
This step can trade a little codon adaptation in one module for less shared sequence across
the set.

`greedy` is the default: it accepts only improvements, and over 18 matched runs it reaches
within 2% of what `anneal` achieves on shared sequence while making a third as many changes.
`anneal` explores further and gives up somewhat more codon adaptation for a small further
gain. `random` is a control — a method that cannot beat it has not been shown to work.

All three are **reproducible for a fixed seed and settings**, not seed-free deterministic:
the candidate replacements come from a seeded generator in every mode.

In [ ]:
#@title Step 4 — optimise the collection {display-mode: "form"}
mode = "greedy"  #@param ["greedy", "anneal", "random"]
max_proposals = 168  #@param {type:"integer"}

optimised = None
if recoded is not None:
    optimised = w.optimise_collection(recoded.artefacts["inventory"], host_codon_table,
                                      "out/collection", mode=mode,
                                      max_proposals=max_proposals, wall_seconds=600)
    print(optimised.summary)

if optimised is not None and "report" in optimised.artefacts:
    search = _json.loads(Path(optimised.artefacts["report"]).read_text(encoding="utf-8"))
    print(f"   adaptation {search['incumbent']['adaptation']} -> "
          f"{search['final']['adaptation']}")
    print(f"   shared k-mers per pair {search['incumbent']['sharing_per_pair']} -> "
          f"{search['final']['sharing_per_pair']}")
    print()
    print("The sharing term steers the search. It is not a recombination probability,")
    print("and lowering it does not necessarily shorten the worst shared tract.")

## Explore junction trade-offs

The kit's internal junction overhangs can be swapped for other four-base sequences encoding
the same protein. Different choices give different predicted assembly fidelity, so there is a
real trade-off to look at rather than a single answer.

⚠️ **Choosing an alternative changes the interface version.** Every target must be recompiled
from the selected inventory, and modules from the previous version cannot be mixed with it.

The result is the **observed** front — nondominated among the candidates actually evaluated.
A bounded search does not enumerate the space, so this is not the globally optimal front.

In [ ]:
#@title Step 5 — explore junction trade-offs {display-mode: "form"}
max_evaluations = 100  #@param {type:"integer"}

front = None
if recoded is not None:
    front = w.explore_interfaces(recoded.artefacts["inventory"], wanted, host_codon_table,
                                 "out/interfaces", max_evaluations=max_evaluations,
                                 wall_seconds=600)
    print(front.summary)
    front = done(front)

if front is not None:
    print()
    print(f"incumbent  : {front.data['incumbent']}")
    print(f"recommended: {front.data['recommended']}")
    print()
    print(f"policy: {front.data['recommendation_policy']}")
    if front.data["fidelity_degenerate"]:
        print()
        print("Every candidate scored the same fidelity -- no trade-off to make here.")
    print()
    print(front.data["selecting_an_alternative"])

## Commit to a choice

Exploring a front does nothing until you pick one. This saves the chosen candidate as a real
inventory and builds the order items from *it*, so what you order is what you chose.

The front is **bound** to the inventory it was measured against. Selecting it against a
different one is refused rather than silently reporting the wrong inventory's numbers.

`incumbent` is a perfectly good choice when the front offers nothing worth an interface change.

In [ ]:
#@title Step 6 — commit to a choice and rebuild {display-mode: "form"}
choice = "recommended"  #@param ["recommended", "incumbent"]

selected = recompiled = order_items = None
if front is not None:
    selected = w.select_interface(recoded.artefacts["inventory"],
                                  front.artefacts["front"], "out/selected",
                                  choice=choice, codon_table=host_codon_table)
    print(selected.summary)
    selected = done(selected)

if selected is not None:
    recompiled = w.load_and_compile(selected.artefacts["inventory"], wanted,
                                    "out/recompiled")
    print(recompiled.summary)
    for failure in recompiled.failures:
        print(f"   could not build {failure['target']}: {failure['reason']}")

    # ok=False here means some substrate is outside the synthesis contract, not that the
    # order failed. All 42 are still written; the flagged ones are named so you can decide.
    order_items = w.order_items_for(selected.artefacts["inventory"], "out/order_items")
    print(order_items.summary)
    for failure in order_items.failures:
        print(f"   {failure['module']}: {failure['reason']}")

> **This route does not take introns into consideration.** The deposited kit modules carry
> none, recoding would silently remove one from a module it rewrote, and nothing here checks
> for them. If you are building for a nuclear host where intron-mediated enhancement matters,
> that is a reason to look at the de novo notebook, not this one.

## Plan the order

Eligibility, pooling and export against a written product profile.

The order items are **assembly-ready substrates**, not bare inserts: each is wrapped so BbsI
releases the level-0 fragment with the correct exposed ends, and each is digest-verified
before export. An A-module insert starts with its own `AATG` fusion site, while the substrate
must expose `CTCA` — ordering the insert would order something that cannot assemble.

**A local check is not vendor approval.** Eligibility and pooling are checked against a
written-down reading of the vendor's published rules; the vendor's current documentation is
always the authority, and no order is placed here.

In [ ]:
#@title Step 7 — eligibility and pool plan {display-mode: "form"}
#@markdown Leave blank for the current published oPools rules. Supply your own profile
#@markdown JSON to check against a different product.
profile_file = ""  #@param {type:"string"}

from ppropt.ordering import CURRENT_OPOOL_50PMOL, load_profile

profile = load_profile(profile_file) if profile_file.strip() else CURRENT_OPOOL_50PMOL

# The order comes from the inventory you selected, never from whatever happens to be left in
# notebook memory. Taking sequences from the synthesis route's `lib` here meant a user could
# explore one inventory and order something unrelated to it.
order = None
if order_items is None or "order_items" not in order_items.artefacts:
    print("Run the cells above first -- the order is built from the selected inventory.")
else:
    items = _json.loads(Path(order_items.artefacts["order_items"]).read_text(
        encoding="utf-8"))
    print(f"ordering {len(items)} assembly-ready substrates from "
          f"{selected.data['to_version']}")
    print(f"exposed ends: {order_items.data['exposed_ends']}")
    order = w.plan_order(items, profile, "out/order")
    print(order.summary)
    for failure in order.failures:
        print(f"   {failure['stage']}: {failure['reason']}")

    print()
    for rule in order.data["eligibility"]["unresolved"]:
        print(f"   unresolved: {rule}")

In [ ]:
#@title Step 7b — where to put the pipette {display-mode: "form"}
from ppropt.viz import kit_plate_svg

if order_items is None:
    print("Run the cells above first.")
else:
    import importlib.resources as _res
    _modules = _json.loads(_res.files("ppropt").joinpath("parts.json")
                           .read_text(encoding="utf-8"))["modules"]
    _wanted = [i["module"] for i in items]
    display(HTML(kit_plate_svg(
        _wanted, _modules,
        title=f"GRASP kit — {len(_wanted)} wells to pick",
        note=f"{len(_wanted)} modules for this build, from inventory "
             f"{selected.data['to_version']}")))

## Take the package

One directory holding the manifest, every task's result, the artefacts and — importantly —
every failure. Artefact paths are recorded relative to the package, so it can be moved or
shared without carrying a machine-specific path along with it.

In [ ]:
#@title Step 8 — download the result package {display-mode: "form"}
done = [r for r in (recoded, compiled, optimised, front, selected, recompiled,
                    order_items, order) if r is not None]
if not done:
    print("Nothing to package yet -- run the cells above.")
else:
    package = w.write_package(done, "out/package",
                              inputs={"table_s1": table_s1, "targets": wanted})
    print(f"wrote {package}")
    summary = _json.loads(Path(package).read_text(encoding="utf-8"))
    print(f"   {len(summary['tasks'])} tasks, ok={summary['ok']}, "
          f"{len(summary['failures'])} failure(s)")
    print()
    print(summary["scope"])

    try:
        import shutil

        from google.colab import files
        shutil.make_archive("ppropt_package", "zip", "out")
        files.download("ppropt_package.zip")
    except ImportError:
        print()
        print("Not running in Colab -- the package is in out/.")

---

## Words used here

Every term this notebook uses, in plain language. Nothing here is a claim about your
design — only what the word means.

| term | what it means |
|---|---|
| **PPR protein** | What this tool designs. A chain of near-identical **repeats**, each one gripping a single RNA base. |
| **target** | The RNA sequence you want the protein to bind. You choose it. |
| **repeat** | One unit of the protein. Two amino acids inside it decide which RNA base it grips, and that mapping is published, so the design is a lookup rather than a search. |
| **architecture (9S / 14S / 19S)** | How many repeats the protein has. It follows from your target's length: 9 bases gives 9S, 14 gives 14S, 19 gives 19S. |
| **codon** | Three DNA letters spelling one amino acid. Most amino acids have several possible codons, so the same protein can be written many different ways. |
| **codon optimisation** | Choosing among those spellings so the DNA is easier to manufacture and easier for your host to read. The protein does not change. |
| **coding sequence (CDS)** | The DNA that spells the protein, start to finish. |
| **fragment** | One of the pieces a gene is split into. Long stretches of DNA are unreliable to synthesise, so the gene is made in a few parts and joined. |
| **overhang** | Four unpaired bases at the end of a fragment. Only its intended partner matches it, so the pieces can join in one order and no other. |
| **`oh5` / `oh3`** | The overhang at a fragment's start and at its end. |
| **junction** | The point where two fragments meet, held together by their matching overhangs. |
| **Golden Gate assembly** | The method used to join the fragments: one enzyme cuts and the pieces are stitched together in a single reaction. |
| **BsaI** | That enzyme. It cuts *outside* the sequence it recognises, which is what lets the overhangs be any four bases you like. |
| **wrapper** | Fixed DNA added to each end of a fragment so the enzyme has something to cut. It is not part of your gene, but it **is** part of what you order. |
| **GC content** | The share of bases that are G or C. Very high or very low makes DNA hard to synthesise. |
| **oligo** | A piece of synthesised DNA. Here, one fragment. |
| **fidelity** | The predicted chance the pieces assemble in the intended order. It comes from published measurements of this enzyme, not from your bench. |
| **coding-span QC** | Checks run on the gene itself. |
| **final artifacts** | Checks run on the exact sequences you would send to a vendor, wrapper included. This is the one that decides orderable. |
| **orderable** | Passed every check this software declares. It is **not** vendor acceptance, and **not** a claim that the protein works. |
| **deposited kit** | The 42 published modules you physically hold. This notebook builds from those rather than designing new DNA. |
| **module** | One part in that kit: a stretch of DNA with fixed ends, ready to join to its neighbours. |
| **interface** | The four-base ends by which modules join. They are held fixed while recoding, so a rewritten module still drops into an assembly built from untouched ones. |
| **recode** | Rewrite a module's DNA for your host's preferred codons. The protein, the length and both interfaces stay exactly as they were. |
| **inventory** | Your saved set of modules after recoding. Every later step works from this rather than from Table S1 again. |
| **pool** | Oligos ordered together in a single tube, which is cheaper than ordering each one separately. |
| **front** | The set of candidates where nothing is strictly better than anything else — improving one property costs you another. A menu of honest trade-offs rather than a ranking, which is why you have to pick one yourself. |
| **level 0** | The smallest assembly unit in this standard: a single part, wrapped so the enzyme releases it with the right ends. Level 0 parts combine into bigger constructs later. |
| **digest check** | A simulated enzyme cut, run before export, confirming each ordered piece really does release the fragment it claims to. |
| **Supplementary Table S1** | The published spreadsheet holding the kit's module sequences. It is not distributed with this package, so you upload it yourself — see `NOTICE.md`. |